In [25]:
import pandas as pd

df = pd.read_csv("Data/movies_metadata.csv", low_memory=False)

# Ambil kolom penting
df = df[['id','title','overview','genres']]
df = df.dropna(subset=['overview'])
df['overview'] = df['overview'].astype(str)

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,2)
)

tfidf_matrix = tfidf.fit_transform(df['overview'])

In [27]:
def tfidf_search(query, top_k=5):
    query_vec = tfidf.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_idx = scores.argsort()[::-1][:top_k]
    return df.iloc[top_idx][['title','overview']]

In [28]:
tfidf_search("robot")


,title,overview
28077,Cody the Robosapien,"At Kinetech Labs, an inventor named Allan Toph..."
32797,"OMG, I'm a Robot!",A sensitive guy finds out he's... a robot.
6425,Eve of Destruction,Eve is a military robot made to look exactly l...
4005,Making Mr. Right,A reclusive scientist builds a robot that look...
7161,Robot Stories,"Four stories including: ""My Robot Baby,"" in wh..."


In [29]:

tfidf_search("love")


,title,overview
43931,A Bela e o Paparazzo,Love on the front page
13513,Nigdy w życiu!,Journey to find true love.
31250,In Lieu of Flowers,Most love stories are about losing love or fin...
36886,Julietta,A dramatic teenage love story set against the ...
36243,"Crouching Tiger, Hidden Dragon: Sword of Destiny","A story of lost love, young love, a legendary ..."


In [30]:
tfidf_search("adventure")

,title,overview
28952,Dinotopia: Quest for the Ruby Sunstone,An Orphaned Boy sets out in search of adventur...
40319,White Wilderness,A fabulous new adventure in exciting entertain...
32819,Pernicious,It was supposed to be an adventure of a lifeti...
34737,Приключения Шерлока Холмса и доктора Ватсона: ...,The Twentieth Century Approaches is a 1986 Sov...
15032,Camille,A twisted honeymoon adventure about a young co...


In [31]:
from rank_bm25 import BM25Okapi

def preprocess(text):
    return text.lower().split()

tokenized_docs = df['overview'].apply(preprocess).tolist()
bm25 = BM25Okapi(tokenized_docs)

In [32]:
def bm25_search(query, top_k=5):
    tokenized_query = preprocess(query)
    scores = bm25.get_scores(tokenized_query)
    top_idx = scores.argsort()[::-1][:top_k]
    return df.iloc[top_idx][['title','overview']]

In [33]:
bm25_search("robot")


,title,overview
28077,Cody the Robosapien,"At Kinetech Labs, an inventor named Allan Toph..."
6425,Eve of Destruction,Eve is a military robot made to look exactly l...
7161,Robot Stories,"Four stories including: ""My Robot Baby,"" in wh..."
4005,Making Mr. Right,A reclusive scientist builds a robot that look...
16873,BURN·E,What lengths will a robot undergo to do his jo...


In [34]:
bm25_search("love")


,title,overview
40093,Out of Love,Out of Love encapsulates the sweltering and de...
36886,Julietta,A dramatic teenage love story set against the ...
31250,In Lieu of Flowers,Most love stories are about losing love or fin...
21930,The Thrill of Brazil,"Steve, revue producer in Rio de Janeiro, is st..."
9161,Dil Se..,The clash between love and ideology is portray...


In [35]:
bm25_search("adventure")

,title,overview
40319,White Wilderness,A fabulous new adventure in exciting entertain...
34737,Приключения Шерлока Холмса и доктора Ватсона: ...,The Twentieth Century Approaches is a 1986 Sov...
25342,Mei and the Kittenbus,Mei has an adventure with a Kittenbus and her ...
32819,Pernicious,It was supposed to be an adventure of a lifeti...
15032,Camille,A twisted honeymoon adventure about a young co...


In [ ]:
from google import genai
import os
from dotenv import load_dotenv
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# fuck you

In [39]:
# get all model
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

In [43]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_dataframe_local(df, column, batch_size=128):
    texts = df[column].fillna("").astype(str).tolist()
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding Progress"):
        batch = texts[i:i + batch_size]
        batch_embeddings = model.encode(batch, show_progress_bar=False)
        embeddings.extend(batch_embeddings)

    return embeddings


df['embedding'] = embed_dataframe_local(df, 'overview')

d:\Users\bsi80269\.conda\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Users\bsi80269\.conda\envs\myenv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\Users\bsi80269\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In or

In [44]:
df.head()

,id,title,overview,genres,embedding
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[0.06343902, 0.001026866, 0.09321016, -0.01494..."
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[0.08630577, 0.0446149, -0.04049633, -0.052532..."
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[-0.10087597, 0.03744183, -0.00092458416, -0.0..."
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","[-0.05541904, -0.01451201, 0.031432502, 0.0424..."
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]","[-0.03138605, -0.0693061, 0.06461947, 0.024486..."


In [ ]:
import chromadb

client = chromadb.Client()
chroma_client = chromadb.Client()
collection = chroma_client.get_collection(name="movies_gemini") if "movies_gemini" in [col.name for col in chroma_client.list_collections()] else chroma_client.create_collection(name="movies_gemini")



In [46]:
# add to collection in chromadb
for idx, row in df.iterrows():
    collection.add(
        ids=[str(row['id'])],
        documents=[row['overview']],
        metadatas=[{"title": row['title']}],
        embeddings=[row['embedding']]
    )


In [47]:
def chroma_search(query, top_k=5):
    query_embedding = model.encode([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['metadatas'][0]

In [48]:
print(chroma_search("Adventure movie with characters named Judy and Peter"))
print(chroma_search("Movies with scenes in New York City"))
print(chroma_search("movie about poet Arthur Rimbaud and Paul Verlaine relationship"))

[{'title': 'Tom and Jerry & The Wizard of Oz'}, {'title': 'Raising Victor Vargas'}, {'title': 'Peter & the Wolf'}, {'title': 'The Wind in the Willows'}, {'title': 'Jumanji'}]
[{'title': 'No More Excuses'}, {'title': 'Kurt Metzger: White Precious'}, {'title': 'Broadway Damage'}, {'title': 'Greenwich Village: Music That Defined a Generation'}, {'title': 'A Most Violent Year'}]
[{'title': 'Total Eclipse'}, {'title': 'Harlan: In the Shadow of Jew Süss'}, {'title': 'Monsieur Verdoux'}, {'title': 'Portrait Werner Herzog'}, {'title': 'Wild Reeds'}]


In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec
load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))


dimension = len(df['embedding'][0])  # assuming all embeddings have the same dimension

index_name = "movies-geminiv2"
if index_name not in [index.name for index in pc.list_indexes()]:
        pc.create_index(
        name=index_name,
        dimension=dimension,  # match embedding dynamically
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
index = pc.index(index_name)



In [85]:
# ✅ clean dataset first
df['title'] = df['title'].fillna("Unknown")

batch_size = 128

for i in range(0, len(df), batch_size):
    end = min(i + batch_size, len(df))

    upsert_data = []
    for j in range(i, end):
        upsert_data.append(
            (
                str(df['id'].iloc[j]),
                df['embedding'].iloc[j].tolist(),
                {"title": str(df['title'].iloc[j])}  # ✅ always string
            )
        )

    index.upsert(vectors=upsert_data, namespace="movies")
    
    print(f"Upserted batch {i} to {end}")

print(index.describe_index_stats())


def pinecone_search(query, top_k=5):
    query_emb = model.encode([query])[0]

    result = index.query(
        vector=query_emb.tolist(),  # ✅ ensure list
        top_k=top_k,
        include_metadata=True,
        namespace="movies"
    )

    return result.matches  # ✅ FIX


Upserted batch 0 to 128
Upserted batch 128 to 256
Upserted batch 256 to 384
Upserted batch 384 to 512
Upserted batch 512 to 640
Upserted batch 640 to 768
Upserted batch 768 to 896
Upserted batch 896 to 1024
Upserted batch 1024 to 1152
Upserted batch 1152 to 1280
Upserted batch 1280 to 1408
Upserted batch 1408 to 1536
Upserted batch 1536 to 1664
Upserted batch 1664 to 1792
Upserted batch 1792 to 1920
Upserted batch 1920 to 2048
Upserted batch 2048 to 2176
Upserted batch 2176 to 2304
Upserted batch 2304 to 2432
Upserted batch 2432 to 2560
Upserted batch 2560 to 2688
Upserted batch 2688 to 2816
Upserted batch 2816 to 2944
Upserted batch 2944 to 3072
Upserted batch 3072 to 3200
Upserted batch 3200 to 3328
Upserted batch 3328 to 3456
Upserted batch 3456 to 3584
Upserted batch 3584 to 3712
Upserted batch 3712 to 3840
Upserted batch 3840 to 3968
Upserted batch 3968 to 4096
Upserted batch 4096 to 4224
Upserted batch 4224 to 4352
Upserted batch 4352 to 4480
Upserted batch 4480 to 4608
Upserted 

In [86]:
print(pinecone_search("Adventure movie with characters named Judy and Peter"))
print(pinecone_search("Movies with scenes in New York City"))
print(pinecone_search("movie about poet Arthur Rimbaud and Paul Verlaine relationship"))

[ScoredVector(id='72972', score=0.515211165, values=[], metadata={'title': 'Tom and Jerry & The Wizard of Oz'}), ScoredVector(id='25461', score=0.507881224, values=[], metadata={'title': 'Raising Victor Vargas'}), ScoredVector(id='14518', score=0.505898476, values=[], metadata={'title': 'Peter & the Wolf'}), ScoredVector(id='59178', score=0.496110946, values=[], metadata={'title': 'The Wind in the Willows'}), ScoredVector(id='8844', score=0.489058495, values=[], metadata={'title': 'Jumanji'})]
[ScoredVector(id='110692', score=0.618826866, values=[], metadata={'title': 'No More Excuses'}), ScoredVector(id='281590', score=0.612108231, values=[], metadata={'title': 'Kurt Metzger: White Precious'}), ScoredVector(id='41638', score=0.583842337, values=[], metadata={'title': 'Broadway Damage'}), ScoredVector(id='158232', score=0.578264236, values=[], metadata={'title': 'Greenwich Village: Music That Defined a Generation'}), ScoredVector(id='241239', score=0.571762085, values=[], metadata={'ti